# 11. AIND Units NWB Export

Build an `AINDUnitsNWBScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDUnitsNWBTask` for each coordinate. The task itself clones [aind-units-nwb](https://github.com/AllenNeuralDynamics/aind-units-nwb) on first use, patches `utils.py` for two neuroconv API renames (`add_electrodes_info_to_nwbfile` → `add_electrodes_to_nwbfile`, `add_units_table_to_nwbfile` → `_add_units_table_to_nwbfile`), seeds its `data/` with the base NWB + `preprocessed/` / `curated/` / `spikesorted/` / `postprocessed/` from the results-collector layout + dispatch `job_*.json` + a synthetic `ecephys_*` session folder, runs `code/run_capsule.py`, and copies the resulting NWB (with a `units` table) into the single config's `coordinate_output_root` (`obi-output/11_aind_units_nwb/grid_scan/0/`).

## Imports and prerequisites

In [ ]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [ ]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "neuroconv", "pynwb", "hdmf-zarr", "aind-nwb-utils",
    ],
    check=True,
)

## Build the scan config

In [ ]:
nwb_input_path = (
    Path.cwd() / "../../../obi-output/10_aind_ecephys_nwb/grid_scan/0"
).resolve()
collected_output_path = (
    Path.cwd() / "../../../obi-output/07_aind_ephys_results_collector/grid_scan/0"
).resolve()
dispatch_output_path = (
    Path.cwd() / "../../../obi-output/01_aind_ephys_dispatch/grid_scan/0"
).resolve()
for p in (nwb_input_path, collected_output_path, dispatch_output_path):
    assert p.exists(), f"{p} not found. Run earlier notebooks first."

scan_config = obi.AINDUnitsNWBScanConfig(
    initialize=obi.AINDUnitsNWBScanConfig.Initialize(
        nwb_input_path=nwb_input_path,
        collected_output_path=collected_output_path,
        dispatch_output_path=dispatch_output_path,
    ),
)

## Generate the grid scan and run the units-NWB task

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/11_aind_units_nwb/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

## Inspect the units table

In [ ]:
from pynwb import NWBHDF5IO

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

nwb_files = sorted(coord_dir.glob("*.nwb"))
print("NWB files:", [p.name for p in nwb_files])

if nwb_files:
    with NWBHDF5IO(str(nwb_files[0]), "r") as io:
        nwb = io.read()
        units = nwb.units
        if units is None:
            print("No units table written.")
        else:
            print("units columns:", list(units.colnames))
            print("num_units:    ", len(units.id))
            df = units.to_dataframe().reset_index()
            keep = [
                c
                for c in [
                    "id",
                    "unit_name",
                    "ks_unit_id",
                    "device_name",
                    "amplitude",
                    "depth",
                    "extremum_channel_index",
                ]
                if c in df.columns
            ]
            print()
            print(df[keep].to_string(index=False))